**To make sure it runs on GPU:**

In [ ]:
!pip uninstall -yq jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt tensorflow-probability
# heads out since jax might drop support for cuda 12, currently (as of Aug 10, 2026), JAX has issues with CUDA 13
# see this post: https://github.com/jax-ml/jax/issues/37923
!pip install -Uq "jax[cuda12]" tfp-nightly blackjax inference_gym optax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 133.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 134.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.9/390.9 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 124.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 158.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.8/175.8 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 110.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow-probability>=0.13.0, which is not installed.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.2 which is incompatible.


In [ ]:
# run those checks if package compatibility is in trouble

# import jax
# import tensorflow_probability as tfp
# import jaxlib

# print("jaxlib:", jaxlib.__version__)
# print("TFP:", tfp.__version__)

# !pip show jax
# !pip show jaxlib
# !pip show blackjax

# import jax.numpy as jnp

# import blackjax

# import tensorflow_probability.substrates.jax as tfp
# import inference_gym.using_jax as gym

# print("JAX:", jax.__version__)
# print("BlackJAX:", blackjax.__version__)
# print("TFP:", tfp.__version__)
# print("ArviZ:", avs.__version__)
# print("Inference Gym imported successfully!")

**Necessary Packages and other settings**

In [ ]:
# GPU set up to accelerate performance
import os
# in case jax eats up my GPU RAM
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_triton_gemm_any=True '
    '--xla_gpu_enable_latency_hiding_scheduler=true '
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import random, jit, vmap, lax
import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
import inference_gym.using_jax as gym
import jaxlib
import blackjax

from blackjax.adaptation.base import get_filter_adapt_info_fn
import optax

# import arviz as az
# import arviz_stats as avs

import warnings
warnings.filterwarnings('ignore')

import psutil

import gc

from google.colab import drive
from matplotlib.lines import Line2D

process = psutil.Process(os.getpid())

def mem(msg):
    print(f"{msg}: {process.memory_info().rss / 1024**2:.1f} MB")

# verification to make sure this is on a GPU
print(jax.devices())
print(jax.default_backend())

[CudaDevice(id=0)]
gpu


In [ ]:
drive.mount('/content/drive')
utility_link = '/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/BJAX_files/PathFinderUtil.py'
with open(utility_link) as f: exec(f.read())

Mounted at /content/drive


In [ ]:
max_warmup = 1000
warmup_window = 100

window_array = np.append(np.repeat(10, 10),
                      np.repeat(warmup_window, max_warmup // warmup_window - 1))

warmup_length = np.repeat(10, len(window_array))
for i in range(len(warmup_length) - 1):
    warmup_length[i + 1] = warmup_length[i] + window_array[i + 1]

# Transition kernel for short regime
repitition = 10
num_chains_short = 2048
num_super_chains = 16

In [ ]:
# quantiles for chi squared with df = 1
chi_up = 3.841459 # 95th quantile for chi squared with df = 1
chi_lo = 0.00393214  # 05th quantile for chi squared with df = 1
tau = 1e-4
M = num_chains_short // num_super_chains
nRhat_lower = np.sqrt(1 + 1 / M + tau)
eps_lower = nRhat_lower - 1
bound = [chi_lo / num_chains_short, chi_up / num_chains_short]
threshold = eps_lower

**MVN with Isotropic Covariate**

In [ ]:
num_dim = 100
# mean array:
mu = jnp.full(num_dim, 1.3)
# covariance matrix:
cov = jnp.eye(num_dim)
target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
    return random.normal(key, shape)

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [ ]:
#simulation part:
Iso_MSE_p_list = []
Iso_RHat_p_list = []

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_p_list,Iso_MSE_p_list,
             mean_benchmark,var_benchmark, num_dim)

Simulation Start: 1707.2 MB
Warmup Length: 10; mean of MSE is: 0.0005332179134711623
Simulation Start: 2687.4 MB
Warmup Length: 20; mean of MSE is: 0.000468497775727883
Simulation Start: 2684.8 MB
Warmup Length: 30; mean of MSE is: 0.000461505725979805
Simulation Start: 2687.6 MB
Warmup Length: 40; mean of MSE is: 0.0005170131335034966
Simulation Start: 2698.4 MB
Warmup Length: 50; mean of MSE is: 0.0005186866037547588
Simulation Start: 2707.6 MB
Warmup Length: 60; mean of MSE is: 0.00048410147428512573
Simulation Start: 2715.9 MB
Warmup Length: 70; mean of MSE is: 0.0004854415892623365
Simulation Start: 2724.6 MB
Warmup Length: 80; mean of MSE is: 0.0004950069705955684
Simulation Start: 2734.2 MB
Warmup Length: 90; mean of MSE is: 0.0005118182161822915
Simulation Start: 2742.7 MB
Warmup Length: 100; mean of MSE is: 0.0004890759009867907
Simulation Start: 2751.2 MB
Warmup Length: 200; mean of MSE is: 0.0004922002553939819
Simulation Start: 2759.9 MB
Warmup Length: 300; mean of MSE is: 

**Save files**

In [ ]:
MSE_p_df = pd.DataFrame(Iso_MSE_p_list)
R_Hat_p_df = pd.DataFrame(Iso_RHat_p_list)

In [ ]:
MSE_p_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/PF_Iso_MSE.pkl"
)
R_Hat_p_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/PF_Iso_Rhat.pkl"
)

**MVN with AR(1) covariance matrix, $\rho = 0.5$**

In [ ]:
num_dim = 100
rho = 0.5
# mean array:
mu = jnp.full(num_dim, 1.3)
# covariance matrix:
cov = [ [rho**(abs(i-j)) for j in range(num_dim)] for i in range(num_dim)]
target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=jnp.array(cov))
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
    return random.normal(key, shape)
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [ ]:
#simulation part:
AR_MSE_p_list = []
AR_RHat_p_list = []

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, AR_RHat_p_list,AR_MSE_p_list,
             mean_benchmark,var_benchmark, num_dim)

Simulation Start: 2911.0 MB
Warmup Length: 10; mean of MSE is: 0.09489055722951889
Simulation Start: 3004.8 MB
Warmup Length: 20; mean of MSE is: 0.057969726622104645
Simulation Start: 3005.8 MB
Warmup Length: 30; mean of MSE is: 0.03692740574479103
Simulation Start: 3006.6 MB
Warmup Length: 40; mean of MSE is: 0.024490538984537125
Simulation Start: 3011.4 MB
Warmup Length: 50; mean of MSE is: 0.016615059226751328
Simulation Start: 3019.8 MB
Warmup Length: 60; mean of MSE is: 0.011347274295985699
Simulation Start: 3027.6 MB
Warmup Length: 70; mean of MSE is: 0.0076964437030255795
Simulation Start: 3035.4 MB
Warmup Length: 80; mean of MSE is: 0.005441886372864246
Simulation Start: 3043.5 MB
Warmup Length: 90; mean of MSE is: 0.003852736670523882
Simulation Start: 3051.7 MB
Warmup Length: 100; mean of MSE is: 0.002775996457785368
Simulation Start: 3061.5 MB
Warmup Length: 200; mean of MSE is: 0.0005114699015393853
Simulation Start: 3069.8 MB
Warmup Length: 300; mean of MSE is: 0.00045076

In [ ]:
MSE_p_df = pd.DataFrame(AR_MSE_p_list)
R_Hat_p_df = pd.DataFrame(AR_RHat_p_list)

In [ ]:
MSE_p_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/PF_AR_MSE.pkl"
)

R_Hat_p_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/PF_AR_Rhat.pkl"
)